 Live survey response analysis

In [ ]:
import json
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import Markdown, display

from survey_responce import rank_llms_by_category, summarize_participants


In [ ]:
config_path = "live-analisys-config.json"
survey_path = "../eval/output/survey.json"

with open(config_path, encoding="utf-8") as config_file:
	config = json.load(config_file)


def fetch_api(path):
	request = Request(
		f"{config['base_url'].rstrip('/')}{path}",
		headers={"Authorization": f"Bearer {config['admin_token']}"},
	)
	with urlopen(request, timeout=30) as api_response:
		return json.load(api_response)


responses = fetch_api("/api/responses")
participants = fetch_api("/api/participants/")


 Responses by participant

In [ ]:
participant_stats = pd.DataFrame(
	summarize_participants(responses, participants, survey_path),
	columns=[
		"participant", "group", "status", "last_page", "saved_at",
		"completed_cases", "incomplete_cases", "complete_rankings",
	],
)
participant_stats = participant_stats[
	participant_stats["participant"].str.match(r"^p\d{2}", na=False)
]
participant_stats["group"] = pd.to_numeric(participant_stats["group"]).astype("Int64")
participant_stats = participant_stats.astype({"last_page": "Int64"})
status_counts = participant_stats["status"].value_counts()
display(Markdown(
	f"**{len(participant_stats)}** participants with responses: "
	f"**{status_counts.get('submitted', 0)}** submitted, **{status_counts.get('draft', 0)}** drafts; "
	f"**{participant_stats['completed_cases'].sum()}** completed cases and "
	f"**{participant_stats['complete_rankings'].sum()}** complete rankings in total."
))
participant_stats.sort_values(["status", "participant"], ascending=[False, True], ignore_index=True)


 Plackett-Luce rankings by category

In [ ]:
# Drafts are autosaved on every change, so rows without both best and worst are skipped.
rankings_by_category = rank_llms_by_category(responses, survey_path, skip_incomplete=True)
for category, rankings in rankings_by_category.items():
	display(Markdown(f"### {category}"))
	display(pd.DataFrame(rankings).set_index("rank"))
if not rankings_by_category:
	display(Markdown("No complete rankings yet."))
